In [75]:
!pip install sentence-transformers faiss-cpu transformers


In [76]:
import os
import re
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [77]:
""" from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    device=0
) """

' from transformers import pipeline\n\ngenerator = pipeline(\n    "text-generation",\n    model="google/flan-t5-base",\n    device=0\n) '

In [78]:
def load_docs(folder):
    docs = []
    for file in os.listdir(folder):
        with open(os.path.join(folder, file), "r", encoding="utf-8") as f:
            docs.append(f.read())
    return docs

documents = load_docs("data")

In [79]:
def parse_gita_text(text):
    """
    Splits full file into slokas based on 'Bg. X.Y'
    """
    pattern = r"(Bg\.\s*\d+\.\d+)"
    parts = re.split(pattern, text)

    chunks = []

    for i in range(1, len(parts), 2):
        ref = parts[i]          # Bg. 18.66
        content = parts[i+1]    # rest of sloka

        match = re.search(r"Bg\.\s*(\d+)\.(\d+)", ref)
        if match:
            chapter = match.group(1)
            verse = match.group(2)
        else:
            chapter, verse = "", ""

        full_text = content.strip()

        chunks.append({
            "text": full_text,
            "chapter": chapter,
            "verse": verse
        })

    return chunks

In [80]:
def chunk_sloka(sloka, chunk_size=500, overlap=100):
    text = sloka["text"]
    chunks = []

    start = 0
    while start < len(text):
        chunk_text = text[start:start+chunk_size]

        chunks.append({
            "text": chunk_text,
            "chapter": sloka["chapter"],
            "verse": sloka["verse"]
        })

        start += chunk_size - overlap

    return chunks


all_chunks = []

for doc in documents:
    slokas = parse_gita_text(doc)
    for sloka in slokas:
        all_chunks.extend(chunk_sloka(sloka))

In [81]:
# 7️⃣ Encode text chunks and build FAISS index
texts = [chunk["text"] for chunk in all_chunks]

# Make sure embedding_model is defined
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode embeddings
embeddings = embedding_model.encode(texts)

# Build FAISS index
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17453.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [82]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(texts)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

In [83]:
import re

def retrieve(query, k=1):
    with open("data/sample.txt", "r", encoding="utf-8") as f:
        text = f.read()

    # Split by sloka
    sloka_sections = re.split(r'(sloka\s*[:.]?\s*Bg\.\s*\d+\.\d+)', text, flags=re.IGNORECASE)
    
    chunks = []
    for i in range(1, len(sloka_sections), 2):
        header = sloka_sections[i].strip()
        content = sloka_sections[i+1].strip()
        
        # Sloka number
        sloka_match = re.search(r'Bg\.\s*(\d+\.\d+)', header, flags=re.IGNORECASE)
        sloka_number = sloka_match.group(1) if sloka_match else "Unknown"
        
        # Book
        book_match = re.search(r'Book:\s*(.*)', text)
        book = book_match.group(1).strip() if book_match else "Bhagavat Gita"
        
        # Add to chunks
        chunks.append({
            "book": book,
            "sloka": sloka_number,
            "text": content
        })

    # Simple keyword-based relevance
    scored = []
    query_lower = query.lower()
    for chunk in chunks:
        score = chunk['text'].lower().count(query_lower)
        if score > 0:
            scored.append((score, chunk))

    scored.sort(reverse=True, key=lambda x: x[0])
    top_chunks = [c[1] for c in scored[:k]]
    
    # If no keyword match, return first k as fallback
    return top_chunks if top_chunks else chunks[:k]

In [84]:
import transformers
print(transformers.__version__)  # should be 5.5.3

5.5.3


In [85]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    device=0  # -1 for CPU
)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 5107.38it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTe

In [86]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large").to("cuda")

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 4327.65it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [87]:
def get_translation(text):
    # Extract translation section only
    match = re.search(r'Translation\s*(.*?)\n\s*Purport', text, flags=re.DOTALL|re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return text  # fallback

In [ ]:
def ask(query, k=1, max_length=128):
    contexts = retrieve(query, k=k)
    if not contexts:
        return "Not found in context"
    
    ctx = contexts[0]
    snippet = get_translation(ctx['text'])  # use only translation
    snippet = " ".join(snippet.split(".")[:3])  # optional: first 2-3 sentences
    
    prompt = f"""
You are an expert on the Bhagavat Gita.
Answer strictly using the Translation from the context below.
Do not invent anything. Always keep the subject exactly as in the text.

Context (from {ctx['book']}, Text #{ctx['TEXT']}):
{snippet}

Question:
{query}

Answer in one clear sentence:
"""
    input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).input_ids.to(model.device)
    outputs = model.generate(
        input_ids,
        max_length=max_length,
        num_beams=4,
        early_stopping=True,
        do_sample=False
    )
    answer_text = tokenizer.decode(outputs[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return f"Text #{ctx['TEXT']} of {ctx['book']}:\n{answer_text}"

In [89]:
print(ask("Whom shall we surrender to?"))

Not found in context
